# Interactive EGFR Cascade Simulation

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import solve_ivp
import ipywidgets as widgets
from IPython.display import display

from model.mechanistic.mechanistic_model import Model
from model.mechanistic.egfr_simplified import model_eqs, PARAM_NAMES, STATE_NAMES, NODE_NAMES

In [ ]:
def light_pulse(t_val, t_args=None):
    """Light on from t_on to t_off, off otherwise."""
    t_on = t_args.get("t_on", 0) if t_args else 0
    t_off = t_args.get("t_off", 5) if t_args else 5
    return 1.0 if t_on <= t_val <= t_off else 0.0

m = Model(
    name="egfr_interactive",
    states=STATE_NAMES,
    parameters=PARAM_NAMES,
    model_definition=model_eqs,
    t_dep="light",
    t_func=light_pulse,
)
system = m.make_numerical()

In [ ]:
DEFAULTS = {
    "k12": 1.0, "k21": 1.0, "K21": 0.5,
    "k34": 1.0, "knfb": 1.0, "k43": 1.0, "K43": 0.5,
    "k56": 1.0, "k65": 1.0, "K65": 0.5,
    "k78": 1.0, "k87": 1.0, "K87": 0.5,
    "f12": 1.0, "f21": 1.0, "F21": 0.5,
}

COLORS = {"RAS": "#e41a1c", "RAF": "#377eb8", "MEK": "#4daf4a", "NFB": "#984ea3", "ERK": "#ff7f00"}

def simulate_and_plot(**kwargs):
    t_end = kwargs.pop("t_end")
    t_on = kwargs.pop("t_on")
    t_off = kwargs.pop("t_off")

    params_vec = np.array([kwargs[p] for p in PARAM_NAMES])
    times = np.linspace(0, t_end, 500)
    y0 = np.zeros(len(STATE_NAMES))
    t_args = {"t_on": t_on, "t_off": t_off}

    sol = solve_ivp(
        lambda t, y: system(t, y, params_vec, light_pulse, t_args),
        [0, t_end], y0, t_eval=times, method="LSODA", rtol=1e-8,
    )

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(10, 6), height_ratios=[1, 4],
                                    sharex=True, gridspec_kw={"hspace": 0.05})

    # Light stimulus
    light_vals = [light_pulse(tv, t_args) for tv in times]
    ax1.fill_between(times, light_vals, alpha=0.3, color="gold")
    ax1.set_ylabel("Light")
    ax1.set_ylim(-0.05, 1.15)
    ax1.set_yticks([0, 1])

    # State trajectories
    if sol.success:
        for i, node in enumerate(NODE_NAMES):
            ax2.plot(times, sol.y[i], label=node, color=COLORS[node], linewidth=2)
        ax2.legend(loc="upper right")
    else:
        ax2.text(0.5, 0.5, f"Solver failed: {sol.message}", transform=ax2.transAxes,
                 ha="center", color="red", fontsize=12)

    ax2.set_xlabel("Time")
    ax2.set_ylabel("Active fraction")
    ax2.set_ylim(bottom=-0.02)
    plt.tight_layout()
    plt.show()

In [ ]:
style = {"description_width": "60px"}

param_sliders = {
    p: widgets.FloatLogSlider(
        value=DEFAULTS[p], base=10, min=-2, max=2, step=0.05,
        description=p, style=style, layout=widgets.Layout(width="350px"),
    )
    for p in PARAM_NAMES
}

sim_sliders = {
    "t_end": widgets.FloatSlider(value=30, min=5, max=100, step=1, description="t_end", style=style,
                                  layout=widgets.Layout(width="350px")),
    "t_on": widgets.FloatSlider(value=0, min=0, max=50, step=0.5, description="t_on", style=style,
                                 layout=widgets.Layout(width="350px")),
    "t_off": widgets.FloatSlider(value=5, min=0, max=50, step=0.5, description="t_off", style=style,
                                  layout=widgets.Layout(width="350px")),
}

# Group sliders by cascade layer
ras_box = widgets.VBox([widgets.Label("RAS"), param_sliders["k12"], param_sliders["k21"], param_sliders["K21"]])
raf_box = widgets.VBox([widgets.Label("RAF"), param_sliders["k34"], param_sliders["knfb"], param_sliders["k43"], param_sliders["K43"]])
mek_box = widgets.VBox([widgets.Label("MEK"), param_sliders["k56"], param_sliders["k65"], param_sliders["K65"]])
erk_box = widgets.VBox([widgets.Label("ERK"), param_sliders["k78"], param_sliders["k87"], param_sliders["K87"]])
nfb_box = widgets.VBox([widgets.Label("NFB"), param_sliders["f12"], param_sliders["f21"], param_sliders["F21"]])
sim_box = widgets.VBox([widgets.Label("Simulation"), *sim_sliders.values()])

left = widgets.VBox([ras_box, raf_box, mek_box])
right = widgets.VBox([erk_box, nfb_box, sim_box])
controls = widgets.HBox([left, right])

out = widgets.interactive_output(simulate_and_plot, {**param_sliders, **sim_sliders})
display(controls, out)